# Introduction
We will use this notebook to explore the data initially, before initiating any trading algorithm.

# Import

In [1]:
# Standard library imports
import datetime as dt
import os

# Third party imports
import pandas as pd
import numpy as np

/Users/mihneapiuaru/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Data Analysis

In [2]:
df_train = pd.read_csv('df_train.csv')

In [7]:
df_train.head()

,date,symbol,open,close,low,high,volume
0,2010-01-04,ACTS,15.13,14.97,14.84,15.28,5459.882
1,2010-01-04,AMWD,7.00,7.12,6.98,7.14,92067.275
2,2010-01-04,ARV,18.27,18.31,18.12,18.47,38034.257
3,2010-01-04,BBY,3.11,3.11,3.09,3.13,40964.820
4,2010-01-04,BCDM,19.78,19.53,19.41,19.90,3646.991


In [8]:
df_train.tail()

,date,symbol,open,close,low,high,volume
100595,2013-12-31,XTG,29.17,28.62,28.99,28.98,42044.703
100596,2013-12-31,YPN,13.88,13.85,13.80,13.96,29030.106
100597,2013-12-31,YRD,11.68,11.57,11.66,11.66,11778.336
100598,2013-12-31,YVNL,2.43,2.40,2.42,2.41,6431.843
100599,2013-12-31,ZQN,21.16,21.42,21.13,21.49,54319.469


In [4]:
df_train.shape

(100600, 7)

In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100600 entries, 0 to 100599
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    100600 non-null  object 
 1   symbol  100600 non-null  object 
 2   open    100600 non-null  float64
 3   close   100600 non-null  float64
 4   low     100600 non-null  float64
 5   high    100600 non-null  float64
 6   volume  100600 non-null  float64
dtypes: float64(5), object(2)
memory usage: 5.4+ MB


In [13]:
print('Number of unique symbols:', len(df_train['symbol'].unique()))
print('Number of unique dates:', len(df_train['date'].unique()))

Number of unique symbols: 100
Number of unique dates: 1006


In [14]:
df_train.describe()

,open,close,low,high,volume
count,100600.000000,100600.000000,100600.000000,100600.000000,1.006000e+05
mean,21.720677,20.363524,20.618948,20.917851,6.660785e+04
std,20.064846,19.852803,18.998291,19.099819,1.057868e+05
min,1.190000,1.100000,1.120000,1.160000,5.868770e+02
25%,7.200000,6.650000,6.890000,6.950000,1.459317e+04
50%,15.090000,13.810000,14.290000,14.690000,3.323060e+04
75%,29.060000,27.300000,28.280000,28.740000,7.313989e+04
max,184.300000,212.430000,182.880000,185.810000,2.990820e+06


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import spearmanr, ttest_1samp

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Data Preparation
Pivot to wide format (dates × symbols). Compute daily returns and 1-day forward returns.

In [1]:
df_train['date'] = pd.to_datetime(df_train['date'])
df_train = df_train.sort_values(['symbol', 'date']).reset_index(drop=True)

# Wide format: rows = dates, columns = symbols
close  = df_train.pivot(index='date', columns='symbol', values='close')
volume = df_train.pivot(index='date', columns='symbol', values='volume')

# Daily simple returns
ret = close.pct_change()

# 1-day forward return
fwd_ret_1d = ret.shift(-1)

print(f"Price matrix shape: {close.shape}")
print(f"Date range: {close.index[0].date()} to {close.index[-1].date()}")
close.head(3)

NameError: name 'pd' is not defined

In [ ]:
# Moving Average Signals
sma5  = close.rolling(window=5,  min_periods=5).mean()
sma20 = close.rolling(window=20, min_periods=20).mean()
sma60 = close.rolling(window=60, min_periods=60).mean()

# Continuous strength versions (more informative than ±1)
ma_cross_cont = (sma5 - sma20) / sma20        # normalised distance fast vs slow MA
price_ma_cont = (close - sma20) / sma20        # normalised distance price vs MA20

# Binary sign versions (for comparison)
ma_cross_sign = np.sign(sma5 - sma20)          # +1 / -1
price_ma_sign = np.sign(close - sma20)

# Momentum Signals
mom10 = close.pct_change(10)   # short-term
mom20 = close.pct_change(20)   # medium-term
mom60 = close.pct_change(60)   # longer-horizon

# Volume-weighted momentum
vol_rank   = volume.rolling(20).mean().rank(axis=1, pct=True)
mom10_vwt  = mom10 * vol_rank   # amplify signal for high-volume stocks

signals = {
    'MA_Cross(5,20)':   ma_cross_cont,
    'Price_vs_MA20':    price_ma_cont,
    'Momentum_10d':     mom10,
    'Momentum_20d':     mom20,
    'Momentum_60d':     mom60,
    'Mom10d_VolWgt':    mom10_vwt,
}
print("Signals constructed:", list(signals.keys()))

In [ ]:
def compute_daily_ic(signal, fwd_ret):
    ic_series = {}
    for date in signal.index.intersection(fwd_ret.index):
        s = signal.loc[date].dropna()
        f = fwd_ret.loc[date].dropna()
        common = s.index.intersection(f.index)
        if len(common) < 10:
            continue
        ic, _ = spearmanr(s[common], f[common])
        if not np.isnan(ic):
            ic_series[date] = ic
    return pd.Series(ic_series)


# Compute IC for every signal
ic_results = {}
for name, sig in signals.items():
    ic = compute_daily_ic(sig, fwd_ret_1d)
    t_stat, _ = ttest_1samp(ic.dropna(), 0)
    ic_results[name] = {
        'Mean IC':  ic.mean(),
        'IC Std':   ic.std(),
        'IC IR':    ic.mean() / ic.std() * np.sqrt(252),
        't-stat':   t_stat,
        'N days':   len(ic),
    }

ic_df = pd.DataFrame(ic_results).T.round(4)
print(ic_df.to_string())

In [ ]:
key_signals = ['MA_Cross(5,20)', 'Momentum_10d', 'Momentum_60d']
key_colors  = ['steelblue', 'darkorange', 'crimson']

fig, axes = plt.subplots(len(key_signals), 1, figsize=(12, 7), sharex=True)

for ax, name, color in zip(axes, key_signals, key_colors):
    ic = compute_daily_ic(signals[name], fwd_ret_1d)
    roll_ic = ic.rolling(63).mean()
    ax.bar(ic.index, ic.values, color=color, alpha=0.25, width=1, label='Daily IC')
    ax.plot(roll_ic.index, roll_ic.values, color=color, lw=1.8, label='63d Rolling IC')
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline(ic.mean(), color=color, lw=1.2, ls='--', label=f'Mean={ic.mean():.4f}')
    ax.set_ylabel('IC')
    ax.set_title(name)
    ax.legend(fontsize=9, loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.suptitle('Daily & Rolling IC (1-day forward return)', y=1.01, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
def long_short_portfolio(signal, fwd_ret, q=0.2):
    records = []
    for date in signal.index.intersection(fwd_ret.index):
        s = signal.loc[date].dropna()
        f = fwd_ret.loc[date].dropna()
        common = s.index.intersection(f.index)
        if len(common) < 10:
            continue
        n = max(1, int(len(common) * q))
        s_c, f_c = s[common], f[common]
        long_ret  = f_c[s_c.nlargest(n).index].mean()
        short_ret = f_c[s_c.nsmallest(n).index].mean()
        records.append({'date': date, 'ls': long_ret - short_ret,
                        'long': long_ret, 'short': short_ret})
    return pd.DataFrame(records).set_index('date')


# Run backtest for all signals
bt_results = {}
summary_rows = []
for name, sig in signals.items():
    pf = long_short_portfolio(sig, fwd_ret_1d)
    ls = pf['ls'].dropna()
    ann_ret  = ls.mean() * 252
    ann_vol  = ls.std()  * np.sqrt(252)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum      = (1 + ls).cumprod()
    max_dd   = (cum / cum.cummax() - 1).min()
    win_rate = (ls > 0).mean()
    bt_results[name] = pf
    summary_rows.append({
        'Signal':       name,
        'Ann. Return':  f"{ann_ret*100:.2f}%",
        'Ann. Vol':     f"{ann_vol*100:.2f}%",
        'Sharpe':       round(sharpe, 3),
        'Max Drawdown': f"{max_dd*100:.2f}%",
        'Win Rate':     f"{win_rate*100:.2f}%",
    })

summary_df = pd.DataFrame(summary_rows).set_index('Signal')
print(summary_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_signals = {
    'MA_Cross(5,20)': ('steelblue', '-'),
    'Momentum_10d':   ('darkorange', '-'),
    'Momentum_60d':   ('crimson', '-'),   # negative = mean-reversion story
    'Mom10d_VolWgt':  ('purple', '--'),
}

for name, (color, ls) in plot_signals.items():
    ls_series = bt_results[name]['ls'].dropna()
    cum = (1 + ls_series).cumprod()
    ax.plot(cum.index, cum.values, color=color, lw=1.8, ls=ls, label=name)

ax.axhline(1, color='black', lw=0.8, ls=':')
ax.set_ylabel('Cumulative Value (starting at 1)')
ax.set_title('Long-Short Quintile Portfolio — Cumulative Returns (no transaction costs)', fontsize=12)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

In [ ]:
horizons = [1, 2, 3, 5, 10, 20]
decay_signals = {
    'MA_Cross(5,20)': ma_cross_cont,
    'Momentum_10d':   mom10,
    'Momentum_60d':   mom60,   # negative IC = mean-reversion story
}

decay_data = {}
for name, sig in decay_signals.items():
    row = []
    for h in horizons:
        # Compound h-day forward return: avoids additive approximation error
        fwd_h = close.shift(-h) / close - 1
        ic_h  = compute_daily_ic(sig, fwd_h)
        row.append(ic_h.mean())
    decay_data[name] = row

decay_df = pd.DataFrame(decay_data, index=[f'h={h}d' for h in horizons])
print("Mean IC at different forecast horizons (compound returns):")
print(decay_df.round(4).to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue', 'darkorange', 'crimson']
for (name, vals), color in zip(decay_data.items(), colors):
    ax.plot(horizons, vals, 'o-', color=color, lw=1.8, label=name)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('Mean IC')
ax.set_title('Signal Decay: Mean IC vs Forecast Horizon', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
yearly_rows = []
for year in [2010, 2011, 2012, 2013]:
    row = {'Year': year}
    for name in ['MA_Cross(5,20)', 'Momentum_10d', 'Momentum_60d']:
        ls_yr = bt_results[name].loc[str(year), 'ls'].dropna()
        ann_r = ls_yr.mean() * 252 * 100
        sharpe_yr = ls_yr.mean() / ls_yr.std() * np.sqrt(252) if ls_yr.std() > 0 else 0
        row[f'{name} Ann%'] = round(ann_r, 1)
        row[f'{name} Sharpe'] = round(sharpe_yr, 2)
    yearly_rows.append(row)

yearly_df = pd.DataFrame(yearly_rows).set_index('Year')
print(yearly_df.to_string())

# Heatmap of annual Sharpe ratios
sharpe_cols = [c for c in yearly_df.columns if 'Sharpe' in c]
sharpe_heat = yearly_df[sharpe_cols].copy()
sharpe_heat.columns = [c.replace(' Sharpe','') for c in sharpe_heat.columns]

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(sharpe_heat.T, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Annual Sharpe Ratio by Signal (Long-Short)')
plt.tight_layout()
plt.show()

In [ ]:
COST = 0.0005   # 5 basis points per unit of turnover

def long_short_with_cost(signal, fwd_ret, q=0.2):
    """Equal-weight L-S quintile with 5bp transaction cost on turnover."""
    records = []
    prev_weights = None
    for date in sorted(signal.index.intersection(fwd_ret.index)):
        s = signal.loc[date].dropna(); f = fwd_ret.loc[date].dropna()
        common = s.index.intersection(f.index)
        if len(common) < 10:
            continue
        n = max(1, int(len(common) * q)); s_c, f_c = s[common], f[common]
        long_idx = s_c.nlargest(n).index; short_idx = s_c.nsmallest(n).index
        weights = pd.Series(0.0, index=common)
        weights[long_idx]  =  1.0 / n
        weights[short_idx] = -1.0 / n
        # Turnover = sum |new_weight - old_weight|
        if prev_weights is not None:
            all_stk = weights.index.union(prev_weights.index)
            turnover = (weights.reindex(all_stk, fill_value=0) -
                        prev_weights.reindex(all_stk, fill_value=0)).abs().sum()
        else:
            turnover = weights.abs().sum()
        ls_gross = f_c[long_idx].mean() - f_c[short_idx].mean()
        cost     = COST * turnover
        records.append({'date': date, 'ls_gross': ls_gross,
                        'turnover': turnover, 'cost': cost, 'ls_net': ls_gross - cost})
        prev_weights = weights
    return pd.DataFrame(records).set_index('date')


# Run for all key signals (use contrarian direction for Momentum_60d)
cost_signals = {
    'MA_Cross(5,20)':          ma_cross_cont,
    'Momentum_10d':            mom10,
    'MeanRev_60d (contrarian)': -mom60,
    'Mom10d_VolWgt':           mom10_vwt,
}

cost_rows = []
pf_net = {}
for name, sig in cost_signals.items():
    pf = long_short_with_cost(sig, fwd_ret_1d)
    g  = pf['ls_gross'].dropna(); n = pf['ls_net'].dropna()
    avg_to      = pf['turnover'].mean()
    daily_cost_bps = pf['cost'].mean() * 10000
    gross_sharpe   = g.mean() / g.std() * np.sqrt(252) if g.std() > 0 else 0
    net_sharpe     = n.mean() / n.std() * np.sqrt(252) if n.std() > 0 else 0
    net_ann        = n.mean() * 252 * 100
    pf_net[name]   = pf
    cost_rows.append({'Signal': name, 'Avg Daily Turnover': round(avg_to, 3),
                      'Avg Cost (bps/day)': round(daily_cost_bps, 2),
                      'Gross Sharpe': round(gross_sharpe, 3),
                      'Net Sharpe': round(net_sharpe, 3),
                      'Net Ann. Return': f"{net_ann:.1f}%"})

cost_df = pd.DataFrame(cost_rows).set_index('Signal')
print(cost_df.to_string())

# Plot cumulative NET returns
fig, ax = plt.subplots(figsize=(12, 5))
colors_net = {'MA_Cross(5,20)': 'steelblue', 'Momentum_10d': 'darkorange',
              'MeanRev_60d (contrarian)': 'crimson', 'Mom10d_VolWgt': 'purple'}
for name, color in colors_net.items():
    ls_net = pf_net[name]['ls_net'].dropna()
    cum = (1 + ls_net).cumprod()
    ax.plot(cum.index, cum.values, color=color, lw=1.8, label=name)
ax.axhline(1, color='black', lw=0.8, ls=':')
ax.set_ylabel('Cumulative Value'); ax.set_title('Post-Cost Cumulative Returns (5bp, daily rebalance)', fontsize=12)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## Conclusion

### Summary of Findings

Numbers reported directly from code output above.

| Signal | Mean IC | t-stat | Gross Sharpe | Net Sharpe (5bp) | Net Ann. Return |
|--------|---------|--------|-------------|-----------------|----------------|
| MA_Cross(5,20) | +0.025 | 4.3 | 0.92 | 0.54 | +7.9% |
| Price_vs_MA20  | +0.022 | 3.7 | 0.45 | — | — |
| Momentum_10d   | +0.027 | 4.8 | 0.89 | **0.07** | +1.0% |
| Momentum_20d   | +0.001 | 0.1 | −0.64 | — | — |
| **MeanRev_60d** (contrarian) | **−0.037** | **−7.2** | **3.36** | **2.94** | **+40.2%** |
| Mom10d_VolWgt  | +0.027 | 5.3 | 1.77 | 0.84 | +10.7% |

---

### Key Signals

**1. Mean Reversion (60d contrarian) — the dominant and most robust signal**  
The 60-day momentum IC is strongly negative (−0.037, t = −7.2) and grows larger at longer horizons — meaning stocks that rallied over the past quarter systematically give back returns. Traded contrarian (buy past losers, short past winners), this delivers:
- Gross Sharpe **3.36**, Net Sharpe **2.94** after 5bp costs
- Annualised net return **+40.2%**
- Positive in **all four years** (2010: +61.7%, 2011: +37.9%, 2012: +39.1%, 2013: +49.0%)
- **Lower turnover** (avg 0.46/day) than momentum strategies → more cost-efficient

**2. Volume-Weighted Momentum (10d) — strongest long-only positive signal**  
Amplifying the 10d momentum signal by volume rank (high-volume stocks get higher weight) raises Sharpe from 0.89 to **1.77** gross. After 5bp costs it falls to **0.84** — still economically meaningful but relies on pre-cost alpha not being fully consumed by daily rebalancing.

**3. MA Cross (5d/20d) — overall positive, but not uniformly strong year-by-year**  
Gross Sharpe 0.92, net Sharpe **0.54** after costs. IC decays quickly (flat beyond h=5 days), confirming this captures short-lived trends. Notably, 2010 produced a negative annual return (−11.6%), reflecting that this signal can underperform in certain market regimes.

**4. Momentum_10d — gross positive, nearly zero net**  
High turnover (avg 0.96/day ≈ 4.82bps daily cost) almost entirely offsets the gross alpha. Net Sharpe drops to **0.07** — this signal is **not tradable** at 5bp costs without further refinement.